### **Mount Drive and Create Folders**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

os.makedirs("/content/drive/MyDrive/project/data", exist_ok=True)
os.makedirs("/content/drive/MyDrive/project/tableau", exist_ok=True)
os.makedirs("/content/drive/MyDrive/project/models", exist_ok=True)

Mounted at /content/drive


### **Start Spark**

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Feature_Engineering") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.2


### **Load Clean Parquet**

In [3]:
data = spark.read.parquet(
"/content/drive/MyDrive/project/data/url_clean.parquet"
)

print("Total records:", data.count())

data.show(5)

Total records: 2396130
+-----+--------------------+
|label|            features|
+-----+--------------------+
| -1.0|(3231961,[1,3,4,5...|
|  1.0|(3231961,[1,3,4,5...|
|  1.0|(3231961,[3,4,5,2...|
| -1.0|(3231961,[1,3,4,5...|
|  1.0|(3231961,[1,3,4,5...|
+-----+--------------------+
only showing top 5 rows


### **Convert Label**

In [4]:
from pyspark.sql.functions import when, col

data = data.withColumn(
    "label",
    when(col("label") == -1, 0).otherwise(1)
)

data.groupBy("label").count().show()

+-----+-------+
|label|  count|
+-----+-------+
|    1| 792145|
|    0|1603985|
+-----+-------+



### **Train Test Validation Split**

In [5]:
train, test = data.randomSplit([0.8, 0.2], seed=42)

train, val = train.randomSplit([0.8, 0.2], seed=42)

print("Train:", train.count())
print("Validation:", val.count())
print("Test:", test.count())

Train: 1533680
Validation: 383308
Test: 479142


### **Scale Features**

In [6]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withStd=True,
    withMean=False
)

scaler_model = scaler.fit(train)

train = scaler_model.transform(train)
test = scaler_model.transform(test)
val = scaler_model.transform(val)

### **Save Processed Data**

In [7]:
train.write.mode("overwrite").parquet(
"/content/drive/MyDrive/project/data/train.parquet"
)

test.write.mode("overwrite").parquet(
"/content/drive/MyDrive/project/data/test.parquet"
)

val.write.mode("overwrite").parquet(
"/content/drive/MyDrive/project/data/val.parquet"
)

### **Save CSV for Tableau Dashboard**

In [8]:
from pyspark.sql.functions import lit

# Train distribution
train_dist = train.groupBy("label").count() \
.withColumn("dataset", lit("Train"))

# Validation distribution
val_dist = val.groupBy("label").count() \
.withColumn("dataset", lit("Validation"))

# Test distribution
test_dist = test.groupBy("label").count() \
.withColumn("dataset", lit("Test"))

# Combine all
combined_dist = train_dist.union(val_dist).union(test_dist)

# Show result
combined_dist.show()

# Save as ONE CSV
combined_dist.toPandas().to_csv(
"/content/drive/MyDrive/project/tableau/data_split_distribution.csv",
index=False
)

+-----+-------+----------+
|label|  count|   dataset|
+-----+-------+----------+
|    1| 506379|     Train|
|    0|1027301|     Train|
|    1| 126980|Validation|
|    0| 256328|Validation|
|    1| 158786|      Test|
|    0| 320356|      Test|
+-----+-------+----------+

